In [10]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import BernoulliNB,GaussianNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures

In [8]:
hr = pd.read_csv(r"../Datasets/HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [16]:
le=LabelEncoder() 
hr['left']=le.fit_transform(hr['left'])
X = hr.drop('left',axis=1)
y=hr['left']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=hr['left'])

In [20]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

scalar = StandardScaler()
X_trn_scl = scalar.fit_transform(X_trn_ohe)
X_tst_scl = scalar.transform(X_tst_ohe)

svm = SVC(C=0.264105, kernel = 'linear',decision_function_shape='ovr')
svm.fit(X_trn_scl, y_train)
y_pred = svm.predict(X_tst_scl)
accuracy_score(y_test,y_pred)


0.7786174705490109

In [23]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")
svm = SVC(C=0.264105, kernel = 'linear',decision_function_shape='ovr')
pipe = Pipeline([('OHE',trans),('SCL', scaler),('SVM',svm)])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
accuracy_score(y_test,y_pred)

0.7786174705490109

### USING KNN

In [29]:
ks=[1,2,3,4,5,6,7]
scores = []
for k in tqdm(ks):
    knn= KNeighborsClassifier(n_neighbors=k)
    pipe = Pipeline([('OHE',trans),('SCL', scaler),('KNN',knn)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    scores.append([k, accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=['ks', 'score'])
df_scores.sort_values('score', ascending=False).head()

  0%|                                                                                            | 0/7 [00:00<?, ?it/s]

 14%|████████████                                                                        | 1/7 [00:03<00:18,  3.00s/it]

 29%|████████████████████████                                                            | 2/7 [00:03<00:06,  1.32s/it]

 43%|████████████████████████████████████                                                | 3/7 [00:03<00:03,  1.28it/s]

 57%|████████████████████████████████████████████████                                    | 4/7 [00:03<00:01,  1.85it/s]

 71%|████████████████████████████████████████████████████████████                        | 5/7 [00:03<00:00,  2.50it/s]

 86%|████████████████████████████████████████████████████████████████████████            | 6/7 [00:03<00:00,  3.20it/s]

100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:03<00:00,  3.89it/s]

100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:03<00:00,  1.80it/s]

,ks,score
0,1,0.964437
1,2,0.960213
2,3,0.948211
3,4,0.946433
5,6,0.945099
